In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import cellrank as cr
import matplotlib.pyplot as plt
import matplotlib as mpl

# ----------------------------
# Basic matplotlib config
# ----------------------------
mpl.rcParams["font.family"] = "Liberation Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------
# User inputs
# ----------------------------
H5AD_PATH = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/05b_trajectory_revisit/adata_CD8_subset_clusters_0_1_2_4.h5ad"
OUTDIR = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/05c_trajectory_rerevisit/results_cellrank2_subset_0_1_2_4"
os.makedirs(OUTDIR, exist_ok=True)

CLUSTER_KEY = "leiden_T_0.6_relabel"     # values: 0/1/2/4 in this subset
RESP_KEY = "Respond"            # "Responder"/"Non-Responder"
EMBED_BASIS = "X_umap"          # use existing UMAP in adata
ROOT_CLUSTER = "4"
ROOT_STRATEGY = "top_naive"
NAIVE_KEY = "score_Naive"   # 你已有的 naive 分数列名；如果没有就用下面方法计算
NAIVE_TOP_FRAC = 0.10       # 前10%
NAIVE_MIN_CAND = 30         # 候选太少时的保底（按你数据量可改）
# GPCCA parameters
N_COMPONENTS = 20
N_STATES_RANGE = (4, 10)        # lets minChi pick
N_TERMINAL = 3                  # top_n terminal macrostates

# Branch hard-label parameters (relative advantage)
DELTA = 0.15  # fp_A - fp_B >= DELTA
EPS   = 0.25  # fp_A (or fp_B) >= EPS

# ----------------------------
# Helpers
# ----------------------------
def savefig(path: str):
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

def pick_root_cell(ad: sc.AnnData, rep: str) -> int:
    """Pick a robust root cell (iroot) using a cluster-defined candidate set."""
    mask_root = (ad.obs[CLUSTER_KEY].astype(str) == str(ROOT_CLUSTER)).values
    idxs = np.where(mask_root)[0]
    if len(idxs) == 0:
        raise ValueError(f"No cells found in ROOT_CLUSTER={ROOT_CLUSTER} for {CLUSTER_KEY}.")

    # ----- candidate selection: top naive within root cluster -----
    if ROOT_STRATEGY == "top_naive":
        if NAIVE_KEY not in ad.obs.columns:
            raise ValueError(f"ROOT_STRATEGY=top_naive but {NAIVE_KEY} not in ad.obs.")
        sub = ad.obs.loc[mask_root, NAIVE_KEY].astype(float)

        # take top 10% (or at least NAIVE_MIN_CAND)
        n_cand = max(int(np.ceil(len(sub) * NAIVE_TOP_FRAC)), NAIVE_MIN_CAND)
        n_cand = min(n_cand, len(sub))
        cand_names = sub.nlargest(n_cand).index
        cand_idxs = np.where(ad.obs_names.isin(cand_names))[0]

        print(f"[INFO] Root candidates: top {n_cand}/{len(sub)} ({NAIVE_TOP_FRAC:.0%}) by {NAIVE_KEY} in cluster {ROOT_CLUSTER}.")
    else:
        # fallback: all cells in root cluster
        cand_idxs = idxs
        print(f"[INFO] Root candidates: all {len(cand_idxs)} cells in cluster {ROOT_CLUSTER}.")

    # ----- pick representation space for medoid -----
    X = ad.obsm[rep][cand_idxs]

    # ----- medoid: most central cell among candidates -----
    x2 = np.sum(X * X, axis=1, keepdims=True)
    D2 = x2 + x2.T - 2 * (X @ X.T)
    D2 = np.maximum(D2, 0.0)
    medoid_local = int(np.argmin(D2.mean(axis=1)))
    iroot = int(cand_idxs[medoid_local])

    print(f"[INFO] Root cell selected by medoid in {rep}: {ad.obs_names[iroot]} (iroot={iroot})")
    return iroot

def ensure_neighbors_for_dpt(ad: sc.AnnData):
    """
    DPT needs connectivities. If neighbors missing, compute from X_scVI if present,
    otherwise from PCA.
    """
    if "neighbors" in ad.uns and "connectivities" in ad.obsp:
        return

    if "X_scVI" in ad.obsm:
        print("[INFO] Computing neighbors using X_scVI.")
        sc.pp.neighbors(ad, use_rep="X_scVI", n_neighbors=50)
    else:
        print("[INFO] Computing PCA + neighbors (X_pca).")
        if "X_pca" not in ad.obsm:
            sc.pp.pca(ad, n_comps=50)
        sc.pp.neighbors(ad, use_rep="X_pca", n_neighbors=50)

def safe_scanpy_violin(ad, keys, groupby, out_png, rotation=45):
    """
    scanpy violin can differ by version; do show=False then save with matplotlib.
    """
    sc.pl.violin(ad, keys=keys, groupby=groupby, show=False, rotation=rotation)
    savefig(out_png)

def infer_lineage_for_cluster(fp_df: pd.DataFrame, ad: sc.AnnData, cluster_value: str) -> str:
    """
    Find which lineage (column in fp_df) is most associated with a given cluster,
    by mean fate probability within that cluster.
    """
    mask = (ad.obs[CLUSTER_KEY].astype(str) == str(cluster_value)).values
    means = fp_df.loc[ad.obs_names[mask]].mean(axis=0)
    return str(means.idxmax())


In [ ]:
ad = sc.read_h5ad(H5AD_PATH)
old_key = "leiden_T_0.6"
new_key = "leiden_T_0.6_relabel"

# 映射：old -> new
mapping = {
    "4": "1",
    "0": "2",
    "1": "3",
    "2": "4",
    "3": "5",
    "5": "6",
    "6": "7",
}

# 确保是字符串（scanpy 的 leiden 通常本来就是字符串）
ad.obs[old_key] = ad.obs[old_key].astype(str)

# 生成新标签；未在 mapping 里的（比如 7/8）会变成 NaN
ad.obs[new_key] = ad.obs[old_key].map(mapping)

# 可选：把没映射到的保留原值（例如 7/8），避免 NaN
ad.obs[new_key] = ad.obs[new_key].fillna(ad.obs[old_key])

# 设为有序类别（按你定义的顺序 1..7，再加上可能的 7/8）
ordered = ["1","2","3","4","5","6","7"]
extras = [c for c in sorted(ad.obs[new_key].unique()) if c not in ordered]
ad.obs[new_key] = pd.Categorical(ad.obs[new_key], categories=ordered + extras, ordered=True)

In [ ]:
# If this file is already subset to 0/1/2/4 you can skip. Keep it safe anyway.
keep = ad.obs[CLUSTER_KEY].astype(str).isin(["1", "2", "3", "4"]).values
ad = ad[keep].copy()
print(f"[INFO] Subset shape: {ad.shape}")
print("[INFO] Subset clusters:\n", ad.obs[CLUSTER_KEY].astype(str).value_counts())

In [ ]:
#ad = sc.read_h5ad(H5AD_PATH)
print(ad)

# If this file is already subset to 0/1/2/4 you can skip. Keep it safe anyway.
keep = ad.obs[CLUSTER_KEY].astype(str).isin(["0", "1", "2", "4"]).values
ad = ad[keep].copy()
print(f"[INFO] Subset shape: {ad.shape}")
print("[INFO] Subset clusters:\n", ad.obs[CLUSTER_KEY].astype(str).value_counts())

In [ ]:
from scipy import sparse
NAIVE_GENES = [
    "TCF7", "LEF1", "CCR7", "IL7R", "LTB", "MAL", "SELL",
    "TRAC", "CD3D", "CD3E"
]

genes = [g for g in NAIVE_GENES if g in ad.var_names]

# 1) 构建 log1p(scvi_norm_expr) layer（安全处理 sparse）
X = ad.layers["norm_expr"]
if sparse.issparse(X):
    X_log = X.copy()
    X_log.data = np.log1p(X_log.data)
else:
    X_log = np.log1p(X)

ad.layers["norm_expr_log1p"] = X_log

# 2) 用这个 layer 算 score
Xbak = ad.X
ad.X = ad.layers["norm_expr_log1p"]
sc.tl.score_genes(ad, gene_list=genes, score_name=NAIVE_KEY, use_raw=False)
ad.X = Xbak

In [ ]:
print(ad)

In [ ]:
out_dir = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/All_results_0123/CD8_Trajectory"
sc.settings.figdir = out_dir
sc.settings.set_figure_params(dpi=200, dpi_save=200)


In [ ]:
#sc.pl.umap(ad, color=NAIVE_KEY, show=False, save="_naive_score.png")
sc.pl.umap(ad, color='leiden_T_0.6_relabel', show=False, save="_CD8_clusters_relabel.png")


In [ ]:
# ----------------------------
# DPT pseudotime
# ----------------------------
#sc.pp.neighbors(ad, use_rep="X_scVI", n_neighbors=50)
sc.pp.neighbors(ad, use_rep="X_scVI", n_neighbors=50)
#ensure_neighbors_for_dpt(ad)
iroot = pick_root_cell(ad, rep="X_scVI")
ad.uns["iroot"] = iroot

# Diffmap + DPT
sc.tl.diffmap(ad)
sc.tl.dpt(ad)  # uses ad.uns["iroot"]
if "dpt_pseudotime" not in ad.obs:
    raise RuntimeError("scanpy did not create ad.obs['dpt_pseudotime'].")

# Quick DPT plots
sc.pl.umap(ad, color="dpt_pseudotime", show=False, save="_dpt_pseudotime.png")
#savefig(os.path.join(OUTDIR, "00_umap_cluster_dpt.png"))

In [ ]:
cluster_col = "leiden_T_0.6_relabel"
pt_col = "dpt_pseudotime"

# 1) 取数据 + 清理
df = ad.obs[[cluster_col, pt_col]].copy()
df = df.dropna(subset=[cluster_col, pt_col])
df[cluster_col] = df[cluster_col].astype(str)

# 2) 按 cluster 的 pseudotime 中位数排序（更像“位置”）
order = (
    df.groupby(cluster_col)[pt_col]
      .median()
      .sort_values()
      .index
      .tolist()
)

data = [df.loc[df[cluster_col] == c, pt_col].to_numpy() for c in order]

# 3) 画 box plot
fig_w = max(8, 0.55 * len(order))  # cluster 多时自动加宽
fig, ax = plt.subplots(figsize=(5, 4.8))

bp = ax.boxplot(
    data,
    labels=order,
    showfliers=False,      # 不显示离群点（更干净）
    patch_artist=True,
    widths=0.6,
    medianprops=dict(linewidth=2),
    boxprops=dict(linewidth=1),
    whiskerprops=dict(linewidth=1),
    capprops=dict(linewidth=1),
)

# 可选：加一点点散点（看密度/离散程度）；点太多就关掉
# rng = np.random.default_rng(0)
# for i, c in enumerate(order, start=1):
#     y = df.loc[df[cluster_col] == c, pt_col].to_numpy()
#     x = i + rng.normal(0, 0.06, size=len(y))
#     ax.plot(x, y, "o", markersize=1.5, alpha=0.15)

ax.set_xlabel(cluster_col)
ax.set_ylabel(pt_col)
ax.set_title(f"Pseudotime distribution by {cluster_col} (ordered by median)")
ax.tick_params(axis="x", labelrotation=45, labelsize=10)
ax.grid(axis="y", linewidth=0.6, alpha=0.4)

plt.tight_layout()
fig.savefig('/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/All_results_0123/CD8_Trajectory/boxplot_dpt_pseudotime_by_cluster.png', dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
CYTO_GENES = ["NKG7","GNLY","GZMB","GZMH","PRF1","FGFBP2","CTSW","KLRD1","KLRB1","TRAC","CD3D","CD3E"]
EXH_GENES  = ["PDCD1","CTLA4","LAG3","HAVCR2","TIGIT","TOX","TOX2","CXCL13","ENTPD1","LAYN","IKZF2"]

# 过滤到数据里存在的基因
ad.X = ad.layers['norm_expr_log1p']
CYTO_GENES = [g for g in CYTO_GENES if g in ad.var_names]
EXH_GENES  = [g for g in EXH_GENES  if g in ad.var_names]
sc.tl.score_genes(ad, gene_list=CYTO_GENES, score_name='cyto_score', use_raw=False)
sc.tl.score_genes(ad, gene_list=EXH_GENES, score_name='exh_score', use_raw=False)

ad.X = Xbak

In [ ]:
df = ad.obs[['dpt_pseudotime', 'exh_score']].copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna()

x = df['dpt_pseudotime'].astype(float).values
y = df['exh_score'].astype(float).values

plt.figure(figsize=(5.2, 4.4))
plt.scatter(x, y, s=6, alpha=0.35)
plt.xlabel('dpt_pseudotime')
plt.ylabel('exh_score')

In [ ]:
df = ad.obs[['dpt_pseudotime', NAIVE_KEY]].copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna()

x = df['dpt_pseudotime'].astype(float).values
y = df[NAIVE_KEY].astype(float).values

plt.figure(figsize=(5.2, 4.4))
plt.scatter(x, y, s=6, alpha=0.35)
plt.xlabel('dpt_pseudotime')
plt.ylabel(NAIVE_KEY)

In [ ]:
sc.pl.umap(ad, color="exh_score", show=False, save="_exh_score.png")
sc.pl.umap(ad, color="cyto_score", show=False, save="_cyto_score.png")


In [ ]:
TIME_DPT   = "dpt_pseudotime"
TIME_NAIVE = "score_Naive"
TIME_CYTO  = "cyto_score"
TIME_EXH   = "exh_score"

In [ ]:
def rank01(x: pd.Series) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    r = x.rank(pct=True)
    return r

# 统一方向：值越大 = 越“晚”
ad.obs["time_dpt"]   = rank01(ad.obs[TIME_DPT])
ad.obs["time_naive"] = rank01(-ad.obs[TIME_NAIVE])   # naive高=早 -> 取负后越大越晚
ad.obs["time_cyto"]  = rank01(ad.obs[TIME_CYTO])
ad.obs["time_exh"]   = rank01(ad.obs[TIME_EXH])


In [ ]:
#sc.pl.umap(ad, color=[CLUSTER_KEY, "time_dpt", "time_naive", "time_cyto", "time_exh"])


In [ ]:
def pick_top_frac_in_cluster(ad, cluster_value, frac=0.05, min_n=50):
    mask = (ad.obs[CLUSTER_KEY].astype(str) == str(cluster_value)).values
    idx = np.where(mask)[0]
    if idx.size == 0:
        raise ValueError(f"No cells in cluster {cluster_value}")
    t = ad.obs['dpt_pseudotime'].values[idx].astype(float)
    thr = np.quantile(t, 1.0 - frac)
    pick = idx[t >= thr]
    if pick.size < min_n:
        # 兜底：取 top min_n（避免太小导致不稳定）
        pick = idx[np.argsort(t)[-min_n:]]
    return np.array(pick, dtype=int)

A_idx = pick_top_frac_in_cluster(ad, cluster_value="2", frac=0.05, min_n=50)  # terminal A from cluster 0
B_idx = pick_top_frac_in_cluster(ad, cluster_value="1", frac=0.05, min_n=50) 

In [ ]:
ad.obs["is_terminal_A_c0top5"] = 0
ad.obs["is_terminal_B_c1top5"] = 0
ad.obs.iloc[A_idx, ad.obs.columns.get_loc("is_terminal_A_c0top5")] = 1
ad.obs.iloc[B_idx, ad.obs.columns.get_loc("is_terminal_B_c1top5")] = 1

In [ ]:
pk = cr.kernels.PseudotimeKernel(ad, time_key='dpt_pseudotime')
pk.compute_transition_matrix()
T=pk.transition_matrix.tocsr()

In [ ]:
n = T.shape[0]

A = np.zeros(n, dtype=float); A[A_idx] = 1.0
B = np.zeros(n, dtype=float); B[B_idx] = 1.0

def kstep_mass(T, indicator, k):
    v = indicator.copy()
    for _ in range(k):
        v = T @ v
    return np.asarray(v).ravel()

for k in [1, 5, 10, 20]:
    uA = kstep_mass(T, A, k)
    uB = kstep_mass(T, B, k)
    ad.obs[f"p_in_A_step{k}"] = uA
    ad.obs[f"p_in_B_step{k}"] = uB
    print(f"k={k}: mean(A)={uA.mean():.4g}, mean(B)={uB.mean():.4g}, frac(A>B)={(uA>uB).mean():.3f}")

# 画一下 k=10 的差值在 UMAP 上（看分叉是否存在）

col = "p_in_A_step10"
eps = 1e-6  # 防止 log(0)

x = ad.obs[col].astype(float).to_numpy()
x = np.clip(x, 0.0, 1.0)

ad.obs[col + "_log"] = np.log(x + eps)

col = "p_in_B_step10"
eps = 1e-6  # 防止 log(0)

x = ad.obs[col].astype(float).to_numpy()
x = np.clip(x, 0.0, 1.0)

ad.obs[col + "_log"] = np.log(x + eps)

ad.obs["k10_prob_branchA"] = ad.obs["p_in_A_step10_log"]
sc.pl.umap(ad, color=["k10_prob_branchA"], show=False)
plt.show()

ad.obs["k10_prob_branchB"] = ad.obs["p_in_B_step10_log"]
sc.pl.umap(ad, color=["k10_prob_branchB"], show=False)
plt.show()
# 看分布
plt.figure(figsize=(5.2,3.6))
plt.hist(ad.obs["k10_prob_branchA"], bins=60, alpha=0.8)
plt.title("Distribution of k=10 reachability to A")
plt.xlabel("k10_prob_branchA")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

x = ad.obs["k10_diff_AminusB"].values.astype(float)

# tau 控制“多快从 0.5 变到两端”，建议用分位数尺度
tau = np.quantile(np.abs(x), 0.7) + 1e-12
s = 1.0 / (1.0 + np.exp(-x / tau))

ad.obs["branch_preference_A_sigmoid"] = s
print("tau:", tau, "saved: branch_preference_A_sigmoid")

In [ ]:
sc.pl.umap(ad, color=["branch_preference_A_sigmoid"], show=False, save="_branch_preference_A_sigmoid.png")
plt.show()

# 看分布
plt.figure(figsize=(5.2,3.6))
plt.hist(ad.obs["branch_preference_A_sigmoid"], bins=60, alpha=0.8)
plt.title("Distribution of k=10 reachability diff (A-B)")
plt.xlabel("p_in_A_step10 - p_in_B_step10")
plt.tight_layout()
plt.show()

In [ ]:
score_key = "branch_preference_A_sigmoid"
out_key   = "branch_AB_0p5"

ad.obs[out_key] = np.where(ad.obs[score_key].values >= 0.46, "branch_A", "branch_B")
ad.obs[out_key] = pd.Categorical(ad.obs[out_key], categories=["branch_A", "branch_B"])

ad.obs[out_key].value_counts()

In [ ]:
import matplotlib.pyplot as plt

X = ad.obsm["X_umap"]
x, y = X[:,0], X[:,1]
lab = ad.obs[out_key].values

plt.figure(figsize=(6,5))
plt.scatter(x, y, s=4, c="lightgrey", alpha=0.4, linewidths=0)
mA = (lab == "branch_A")
plt.scatter(x[mA], y[mA], s=6, c="red", alpha=0.9, linewidths=0)
plt.title("Hard branch assignment (>=0.5 = A)")
plt.xlabel("UMAP1"); plt.ylabel("UMAP2")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "hard_branch_A.png"), dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(6,5))
plt.scatter(x, y, s=4, c="lightgrey", alpha=0.4, linewidths=0)
mB = (lab == "branch_B")
plt.scatter(x[mB], y[mB], s=6, c="blue", alpha=0.9, linewidths=0)
plt.title("Hard branch assignment (<0.5 = B)")
plt.xlabel("UMAP1"); plt.ylabel("UMAP2")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "hard_branch_B.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.stats import rankdata, norm
key_out = "branch_preference_A_sigmoid_z"

x = ad.obs['branch_preference_A_sigmoid'].astype(float).values
m = np.isfinite(x)

z = np.full_like(x, np.nan, dtype=float)
r = rankdata(x[m], method="average")  # 1..n
n = r.size

# Blom / rankit transform: (r - 0.375)/(n + 0.25) 更稳定，避免无穷大
p = (r - 0.375) / (n + 0.25)
z[m] = norm.ppf(p)   # -> approx N(0,1)

ad.obs[key_out] = z

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

UMAP = "X_umap"
keyA = "is_terminal_A_c0top5"
keyB = "is_terminal_B_c1top5"

X = ad.obsm[UMAP]
x, y = X[:, 0], X[:, 1]

# 转成bool，兼容0/1、True/False、"True"/"False"
A = ad.obs[keyA].astype(bool).values
B = ad.obs[keyB].astype(bool).values

plt.figure(figsize=(10, 7.5))

# 1) background
plt.scatter(x, y, s=10, c="lightgrey", alpha=0.55, linewidths=0)

# 2) terminal A (red)
plt.scatter(
    x[A], y[A],
    s=80, c="red", edgecolors="black", linewidths=1.2,
    label="A (c0 top5%)"
)

# 3) terminal B (blue)
plt.scatter(
    x[B], y[B],
    s=80, c="dodgerblue", edgecolors="black", linewidths=1.2,
    label="B (c1 top5%)"
)

plt.title("terminal states")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "terminal_states.png"), dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----- data -----
X = ad.obsm["X_umap"].astype(float)
x = X[:, 0]
y = X[:, 1]
f = ad.obs["branch_preference_A_sigmoid"].values.astype(float)  # 0~1

# ----- grid -----
bins = 26
gx = np.linspace(np.quantile(x, 0.02), np.quantile(x, 0.98), bins)
gy = np.linspace(np.quantile(y, 0.02), np.quantile(y, 0.98), bins)
GX, GY = np.meshgrid(gx, gy, indexing="xy")

def grid_local_mean(x, y, val, GX, GY, radius_frac=0.12, min_points=80):
    xmin, xmax = np.quantile(x, [0.02, 0.98])
    ymin, ymax = np.quantile(y, [0.02, 0.98])
    rx = (xmax - xmin) * radius_frac
    ry = (ymax - ymin) * radius_frac
    r2 = rx*rx + ry*ry

    V = np.full(GX.shape, np.nan, dtype=float)
    ok = np.zeros(GX.shape, dtype=bool)
    for i in range(GX.shape[0]):
        for j in range(GX.shape[1]):
            x0, y0 = GX[i, j], GY[i, j]
            dx = x - x0
            dy = y - y0
            d2 = dx*dx + dy*dy
            sel = d2 <= r2
            if sel.sum() < min_points:
                continue
            w = np.exp(-d2[sel] / (0.5*r2))
            V[i, j] = np.sum(w * val[sel]) / (np.sum(w) + 1e-12)
            ok[i, j] = True
    return V, ok

# 网格上的平均 f（用于更稳定的梯度）
Fgrid, okF = grid_local_mean(x, y, f, GX, GY, radius_frac=0.12, min_points=80)

# ----- gradient on grid -----
# np.gradient: 对应 axis0=gy, axis1=gx
dFy, dFx = np.gradient(Fgrid, gy, gx)  # dF/dy, dF/dx

ok = okF & np.isfinite(dFx) & np.isfinite(dFy) & np.isfinite(Fgrid)

# ----- make arrows show up also in the middle -----
# 软权重：在 Fgrid 接近 0.5（主干）时也留一点箭头，不要完全没方向
# 如果你希望中间更强，把 tau 调大一点
tau = 0.12
s = np.tanh((Fgrid - 0.5) / tau)  # [-1,1]，中间接近0但不完全“死”

U = s * dFx
V = s * dFy

# 统一缩放，避免极端长箭头
mag = np.sqrt(U**2 + V**2)
cap = np.nanquantile(mag[ok], 0.90)
scale = 0.9 / (cap + 1e-12)   # 0.6~1.2 调整箭头总体长度
U *= scale
V *= scale

# ----- plot -----
plt.figure(figsize=(6.4, 5.4))
plt.scatter(x, y, s=4, c=f, alpha=0.75, linewidths=0)
plt.colorbar(label="branch_preference_A_sigmoid")

plt.quiver(GX[ok], GY[ok], U[ok], V[ok],
           angles="xy", scale_units="xy", scale=1.0,
           width=0.003, alpha=0.9, color="black")

plt.title("UMAP + arrows from branch_preference_A_sigmoid")
plt.xlabel("UMAP1"); plt.ylabel("UMAP2")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "UMAP_arrows_branch_preference_A_sigmoid.png"), dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
term = ad.obs[keyA].astype(bool).values | ad.obs[keyB].astype(bool).values

def stay_prob(T, term_mask, k=1):
    Tk = T.copy()
    for _ in range(k-1):
        Tk = Tk @ T
    # p_stay(i) = sum_j Tk[i,j] for j in terminal set
    p = np.asarray(Tk[:, term_mask].sum(axis=1)).ravel()
    return p

p1  = stay_prob(T, term, k=1)
p10 = stay_prob(T, term, k=10)

# 画 UMAP：p10
X = ad.obsm[UMAP].astype(float)
x, y = X[:,0], X[:,1]

plt.figure(figsize=(6.6, 5.6))
plt.scatter(x, y, s=6, c=p10, alpha=0.9, linewidths=0)
plt.colorbar(label="P(stay in terminal set) after 10 steps")
plt.scatter(x[term], y[term], s=45, facecolors="none", edgecolors="black", linewidths=1.2)
plt.title("k=10 terminal-set staying probability (black circles = terminal cells)")
plt.xlabel("UMAP1"); plt.ylabel("UMAP2")
plt.tight_layout()
plt.show()

# 再用直方图对比 terminal vs 非terminal（k=1 和 k=10）
plt.figure(figsize=(7.2, 4.2))
plt.hist(p1[~term], bins=60, alpha=0.5, label="non-terminal (k=1)")
plt.hist(p1[term],  bins=60, alpha=0.7, label="terminal (k=1)")
plt.title("Stay prob to terminal set (k=1)")
plt.legend(frameon=False); plt.tight_layout(); plt.show()

plt.figure(figsize=(7.2, 4.2))
plt.hist(p10[~term], bins=60, alpha=0.5, label="non-terminal (k=10)")
plt.hist(p10[term],  bins=60, alpha=0.7, label="terminal (k=10)")
plt.title("Stay prob to terminal set (k=10)")
plt.legend(frameon=False); plt.tight_layout(); plt.show()

In [ ]:
from scipy.stats import rankdata
BRANCH_KEY = "branch_preference_A_sigmoid"   # 你的A/B/other标签
DPT_KEY    = "dpt_pseudotime"
NAIVE_KEY  = "score_Naive"
CYTO_KEY   = "cyto_score"
EXH_KEY    = "exh_score"

maskA = ad.obs[BRANCH_KEY].values >= 0.46 
maskB = ad.obs[BRANCH_KEY].values < 0.46

# --- raw values ---
branch_A_dpt   = ad.obs.loc[maskA, DPT_KEY].astype(float).values
branch_B_dpt   = ad.obs.loc[maskB, DPT_KEY].astype(float).values

branch_A_naive = ad.obs.loc[maskA, NAIVE_KEY].astype(float).values
branch_B_naive = ad.obs.loc[maskB, NAIVE_KEY].astype(float).values

branch_A_cyto  = ad.obs.loc[maskA, CYTO_KEY].astype(float).values
branch_B_cyto  = ad.obs.loc[maskB, CYTO_KEY].astype(float).values

branch_A_exh   = ad.obs.loc[maskA, EXH_KEY].astype(float).values
branch_B_exh   = ad.obs.loc[maskB, EXH_KEY].astype(float).values

print("nA =", maskA.sum(), "nB =", maskB.sum())
branch_A_transit_prob   = ad.obs.loc[maskA, 'k10_diff_AminusB'].astype(float).values
branch_B_transit_prob   = ad.obs.loc[maskB, 'k10_diff_AminusB'].astype(float).values


In [ ]:
def to_uniform_01(x: np.ndarray) -> np.ndarray:
    """Rank-based mapping to ~Uniform(0,1)."""
    x = np.asarray(x, dtype=float)
    n = x.size
    if n <= 1:
        return np.zeros_like(x)
    r = rankdata(x, method="average")  # 1..n
    return (r - 1) / (n - 1)

branch_A_dpt_u = to_uniform_01(branch_A_dpt)
branch_B_dpt_u = to_uniform_01(branch_B_dpt)

print(branch_A_dpt_u.min(), branch_A_dpt_u.max(), " | ", branch_B_dpt_u.min(), branch_B_dpt_u.max())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def binned_mean_curve(x, y, n_bins=60, min_n=15):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    bins = np.linspace(0.0, 1.0, n_bins + 1)

    xc, yc = [], []
    for i in range(n_bins):
        sel = (x >= bins[i]) & (x < bins[i+1]) if i < n_bins-1 else (x >= bins[i]) & (x <= bins[i+1])
        if sel.sum() < min_n:
            continue
        xc.append(0.5 * (bins[i] + bins[i+1]))
        yc.append(np.mean(y[sel]))
    return np.array(xc), np.array(yc)

def smooth_rolling(y, win=9):
    # win 必须是奇数更好看；win 越大越平滑（建议 9~21）
    s = pd.Series(y)
    return s.rolling(window=win, center=True, min_periods=max(3, win//3)).mean().to_numpy()

def plot_two_lines(xA, yA, xB, yB, title, ylabel, win=11):
    yA_s = smooth_rolling(yA, win=win)
    yB_s = smooth_rolling(yB, win=win)

    plt.figure(figsize=(6.2, 4.8))
    plt.plot(xA, yA_s, linewidth=2.8, color="red",  label="A")
    plt.plot(xB, yB_s, linewidth=2.8, color="blue", label="B")
    plt.xlabel("within-branch uniform pseudotime (0-1)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{title}.png"), dpi=200, bbox_inches="tight")
    plt.show()

# 例子：naive
xA, yA = binned_mean_curve(branch_A_dpt_u, branch_A_naive, n_bins=70, min_n=20)
xB, yB = binned_mean_curve(branch_B_dpt_u, branch_B_naive, n_bins=70, min_n=20)
plot_two_lines(xA, yA, xB, yB, "Naive score trend (smoothed)", "score_Naive", win=13)

# cyto
xA, yA = binned_mean_curve(branch_A_dpt_u, branch_A_cyto, n_bins=70, min_n=20)
xB, yB = binned_mean_curve(branch_B_dpt_u, branch_B_cyto, n_bins=70, min_n=20)
plot_two_lines(xA, yA, xB, yB, "Cytotoxic score trend (smoothed)", "cyto_score", win=13)

# exh
xA, yA = binned_mean_curve(branch_A_dpt_u, branch_A_exh, n_bins=70, min_n=20)
xB, yB = binned_mean_curve(branch_B_dpt_u, branch_B_exh, n_bins=70, min_n=20)
plot_two_lines(xA, yA, xB, yB, "Exhaustion score trend (smoothed)", "exh_score", win=13)

xA, yA = binned_mean_curve(branch_A_dpt_u, branch_A_transit_prob, n_bins=70, min_n=20)
xB, yB = binned_mean_curve(branch_B_dpt_u, branch_B_transit_prob, n_bins=70, min_n=20)
plot_two_lines(xA, yA, xB, yB, "Transit probability trend (smoothed)", "k10_diff_AminusB", win=13)


In [ ]:
import numpy as np
import scipy.sparse as sp

key_in  = "k10_diff_AminusB"
key_out = "k10_diff_AminusB_smooth1"

assert key_in in ad.obs.columns
assert "connectivities" in ad.obsp, "Run sc.pp.neighbors(...) first."

C = ad.obsp["connectivities"].tocsr()

x = ad.obs[key_in].values.astype(float)
x0 = x.copy()

# 归一化：每行和为 1
deg = np.asarray(C.sum(axis=1)).ravel()
deg[deg == 0] = 1.0

x_nb = (C @ x0) / deg

# 混合：alpha 越大越依赖邻居（越平滑）
alpha = 1
x_smooth = (1 - alpha) * x0 + alpha * x_nb

ad.obs[key_out] = x_smooth
print("saved:", key_out)


In [ ]:
import numpy as np
import pandas as pd

diff = ad.obs["k10_diff_AminusB_smoothK"].values

# 自动阈值：让大约 20% 的细胞被分到 A 或 B（你也可以调 0.75/0.85）
q = 0.3
delta = np.quantile(np.abs(diff), q)
print("auto delta =", delta)

label = np.array(["other"] * ad.n_obs, dtype=object)
label[diff >=  delta] = "branch_A"
label[diff <= -delta] = "branch_B"

ad.obs["branch_k10"] = pd.Categorical(label, categories=["branch_A","branch_B","other"])
ad.obs["branch_k10"].value_counts()


In [ ]:
X = ad.obsm["X_umap"]
x, y = X[:, 0], X[:, 1]

maskA = (ad.obs["branch_k10"] == "branch_A").values
maskB = (ad.obs["branch_k10"] == "branch_B").values

# A 红
plt.figure(figsize=(5.8, 5.0))
plt.scatter(x[~maskA], y[~maskA], s=6, c="lightgrey", alpha=0.25, linewidths=0)
plt.scatter(x[maskA],  y[maskA],  s=10, c="red", alpha=0.90, linewidths=0)
plt.title(f"Branch A by k-step (k=10), |diff| >= {delta:.3g}")
plt.xlabel("UMAP1"); plt.ylabel("UMAP2")
plt.tight_layout()
plt.show()

# B 蓝
plt.figure(figsize=(5.8, 5.0))
plt.scatter(x[~maskB], y[~maskB], s=6, c="lightgrey", alpha=0.25, linewidths=0)
plt.scatter(x[maskB],  y[maskB],  s=10, c="blue", alpha=0.90, linewidths=0)
plt.title(f"Branch B by k-step (k=10), |diff| >= {delta:.3g}")
plt.xlabel("UMAP1"); plt.ylabel("UMAP2")
plt.tight_layout()
plt.show()


In [ ]:
RESP_KEY = "Respond"
tab = pd.crosstab(ad.obs["branch_k10"], ad.obs[RESP_KEY], normalize="columns")
display(tab)
CLUSTER_KEY = "leiden_T_0.6"

for cl in sorted(ad.obs[CLUSTER_KEY].astype(str).unique()):
    sub = ad[ad.obs[CLUSTER_KEY].astype(str)==cl]
    if sub.n_obs < 50:
        continue
    print("cluster", cl, "n=", sub.n_obs)
    display(sub.obs.groupby(RESP_KEY)["k10_diff_AminusB_smoothK"].describe()[["mean","50%","std","min","max"]])


In [ ]:
import pandas as pd

score_keys = ["dpt_pseudotime", "score_Naive", "cyto_score", "exh_score"]
score_keys = [k for k in score_keys if k in ad.obs.columns]

summ_branch = ad.obs.groupby("branch_k10")[score_keys].agg(["mean","median","count"])
display(summ_branch)

In [ ]:
X = ad.obsm["X_umap"].astype(float)
x, y = X[:, 0], X[:, 1]

f = ad.obs["k10_diff_AminusB_smoothK"].values.astype(float)  # 标量场

In [ ]:
print(ad.obs['branch_AB_0p5'].value_counts())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, fisher_exact

CLUSTER_KEY = "leiden_T_0.6"
RESP_KEY    = "Respond"


In [ ]:
df1 = ad.obs.loc[:, [RESP_KEY, "branch_AB_0p5"]].copy()
tab1 = pd.crosstab(df1["branch_AB_0p5"], df1[RESP_KEY], normalize="columns")  # 每个Respond列归一化
display(tab1)

ax = tab1.T.plot(kind="bar", figsize=(6,4))
ax.set_ylabel("Fraction within Respond group")
ax.set_title("Branch composition (A/B) within Responder vs Non-Responder")
plt.xticks(rotation=0)

ax.legend(title="branch_AB_0p5", loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

p = 0.0
chi2 = 1912.519245430069
dof = 1
p_txt = "<1e-300" if p == 0 or p < 1e-300 else f"{p:.2e}"
ax.text(0.02, 0.98, f"Chi-square: χ²={chi2:.1f}, dof={dof}, p={p_txt}",
        transform=ax.transAxes, ha="left", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.75, edgecolor="none"))

plt.tight_layout()
plt.savefig(os.path.join(out_dir, "branch_composition_A_B_within_Responder_vs_Non-Responder.png"),
            dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
tab1_count = pd.crosstab(df1["branch_AB_0p5"], df1[RESP_KEY])
display(tab1_count)

# 统一顺序
tab1_count = tab1_count.reindex(index=["branch_A","branch_B"])
# Respond列名可能顺序不固定
cols = [c for c in ["Responder","Non-Responder"] if c in tab1_count.columns] + \
       [c for c in tab1_count.columns if c not in ["Responder","Non-Responder"]]
tab1_count = tab1_count[cols]

# Fisher/Chi2
arr = tab1_count.values
use_fisher = (arr.shape==(2,2)) and (arr.min() < 5)
if use_fisher:
    oddsratio, p = fisher_exact(arr)
    print("[Test] Fisher exact p =", p, "oddsratio =", oddsratio)
else:
    chi2, p, dof, exp = chi2_contingency(arr)
    print("[Test] Chi-square p =", p, "chi2 =", chi2, "dof =", dof)
    print("Expected:\n", pd.DataFrame(exp, index=tab1_count.index, columns=tab1_count.columns))


In [ ]:
cluster_target = "4"
mask_c4 = (ad.obs[CLUSTER_KEY].astype(str) == str(cluster_target)).values
df2 = ad.obs.loc[mask_c4, [RESP_KEY, "branch_AB_0p5"]].copy()

tab2_count = pd.crosstab(df2["branch_AB_0p5"], df2[RESP_KEY])
display(tab2_count)

# 2.1 在 cluster4 内：每个 Respond 组里 A/B 的比例
tab2_colnorm = pd.crosstab(df2["branch_AB_0p5"], df2[RESP_KEY], normalize="columns")
display(tab2_colnorm)

ax = tab2_colnorm.T.plot(kind="bar", figsize=(6,4))
ax.set_ylabel("Fraction within Respond group (cluster 4)")
ax.set_title("Cluster 4: Branch composition (A/B) within Responder vs Non-Responder")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 2.2 在 cluster4 内：每个 Branch 里 R/NR 的比例
tab2_rownorm = pd.crosstab(df2["branch_AB_0p5"], df2[RESP_KEY], normalize="index")
display(tab2_rownorm)

ax = tab2_rownorm.plot(kind="bar", figsize=(6,4))
ax.set_ylabel("Fraction within Branch (cluster 4)")
ax.set_title("Cluster 4: Respond composition within Branch A/B")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
tab2_count = tab2_count.reindex(index=["branch_A","branch_B"])
cols = [c for c in ["Responder","Non-Responder"] if c in tab2_count.columns] + \
       [c for c in tab2_count.columns if c not in ["Responder","Non-Responder"]]
tab2_count = tab2_count[cols]

arr = tab2_count.values
use_fisher = (arr.shape==(2,2)) and (arr.min() < 5)
if use_fisher:
    oddsratio, p = fisher_exact(arr)
    print("[Cluster 4 Test] Fisher exact p =", p, "oddsratio =", oddsratio)
else:
    chi2, p, dof, exp = chi2_contingency(arr)
    print("[Cluster 4 Test] Chi-square p =", p, "chi2 =", chi2, "dof =", dof)
    print("Expected:\n", pd.DataFrame(exp, index=tab2_count.index, columns=tab2_count.columns))


In [ ]:
pseudo_col = "dpt_pseudotime"
pseudo_u_col = "dpt_pseudotime_uniform"

x = ad.obs[pseudo_col].astype(float).values
m = np.isfinite(x)

# 分位数映射：rank -> [0,1]
r = pd.Series(x[m]).rank(method="average").values  # 1..n
u = (r - 1) / (len(r) - 1 + 1e-12)

ad.obs[pseudo_u_col] = np.nan
ad.obs.loc[ad.obs.index[m], pseudo_u_col] = u

In [ ]:
DE_DIR = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells"
DE_4TO1 = os.path.join(DE_DIR, "DE_cluster1_vs_cluster4.full.csv")
DE_4TO2 = os.path.join(DE_DIR, "DE_cluster2_vs_cluster4.full.csv")
DE_4_RVSNR = os.path.join(DE_DIR, "DE_cluster4_R_vs_NR.full.csv")
DE_1_RVSNR = os.path.join(DE_DIR, "DE_cluster1_R_vs_NR.full.csv")

In [ ]:
OUTDIR = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/All_results_0123/perturb_seq"
os.makedirs(OUTDIR, exist_ok=True)

In [ ]:
ad.obs["RNR"] = ad.obs['Respond'].astype(str).map({"Responder": "R", "Non-Responder": "NR"})


In [ ]:
def _get_matrix(ad, genes, layer="norm_expr_log1p", log1p=True):
    # 取表达矩阵 (n_cells, n_genes)
    if layer is None:
        X = ad[:, genes].X
    else:
        if layer not in ad.layers:
            raise ValueError(f"layer '{layer}' not in ad.layers. Available: {list(ad.layers.keys())}")
        X = ad[:, genes].layers[layer]

    if sparse.issparse(X):
        X = X.toarray()
    X = np.asarray(X)

    if log1p:
        X = np.log1p(X)
    return X

def signature_score_from_de(
    ad,
    de_csv,
    score_name,
    layer="norm_expr_log1p",
    log1p=True,
    fdr=0.05,
    abs_lfc=0.25,
    topk=300
):
    """
    score(cell) = sum_g [ sign(logFC_g)*|logFC_g| * expr(cell,g) ]
    where g are selected from DE table (filtered by FDR, abs_lfc, topk).
    """
    de = pd.read_csv(de_csv)
    de = de.dropna(subset=["names", "logfoldchanges"]).copy()
    de["names"] = de["names"].astype(str)

    if "pvals_adj" in de.columns:
        de = de[de["pvals_adj"] <= fdr]
    de = de[de["logfoldchanges"].abs() >= abs_lfc]
    de["abs_lfc"] = de["logfoldchanges"].abs()
    de = de.sort_values("abs_lfc", ascending=False).head(topk)

    genes = [g for g in de["names"].tolist() if g in ad.var_names]
    if len(genes) < 30:
        raise ValueError(f"Too few genes after filtering for {score_name}: {len(genes)}. Try lower abs_lfc or larger topk.")

    de = de.set_index(de["names"])
    w = np.sign(de.loc[genes, "logfoldchanges"].values) * np.abs(de.loc[genes, "logfoldchanges"].values)

    X = _get_matrix(ad, genes, layer=layer, log1p=log1p)
    score = X @ w  # (n_cells,)

    ad.obs[score_name] = score
    return len(genes)

# 计算 4 个 signature score（你也可以只先算 2 个）
n_4to1 = signature_score_from_de(ad, DE_4TO1, "sig_4to1", layer="norm_expr_log1p", log1p=True, fdr=0.05, abs_lfc=0.25, topk=300)
n_4to2 = signature_score_from_de(ad, DE_4TO2, "sig_4to2", layer="norm_expr_log1p", log1p=True, fdr=0.05, abs_lfc=0.25, topk=300)

# NR->R 两个可以先算其中一个（你要图1建议加上）
n_4nr2r = signature_score_from_de(ad, DE_4_RVSNR, "sig_4_NR2R", layer="norm_expr_log1p", log1p=True, fdr=0.05, abs_lfc=0.25, topk=300)
n_1nr2r = signature_score_from_de(ad, DE_1_RVSNR, "sig_1_NR2R", layer="norm_expr_log1p", log1p=True, fdr=0.05, abs_lfc=0.25, topk=300)

print("n_genes used:", {"sig_4to1": n_4to1, "sig_4to2": n_4to2, "sig_4_NR2R": n_4nr2r, "sig_1_NR2R": n_1nr2r})


In [ ]:
def plot_score_vs_pseudotime(ad, score_col, pseudo_col, group_col="RNR",
                             bins=60, min_bin_n=25, alpha=0.12, title=None,
                             save_prefix=None):
    x = ad.obs[pseudo_col].values.astype(float)
    y = ad.obs[score_col].values.astype(float)
    g = ad.obs[group_col].astype(str).values

    m = np.isfinite(x) & np.isfinite(y) & pd.notna(g)
    x, y, g = x[m], y[m], g[m]

    plt.figure(figsize=(6.2, 4.2))
    for grp in sorted(pd.unique(g)):
        idx = (g == grp)
        plt.scatter(x[idx], y[idx], s=6, alpha=alpha, label=grp)

        # bin-mean smooth
        edges = np.linspace(np.nanmin(x), np.nanmax(x), bins + 1)
        b = np.digitize(x[idx], edges)  # 1..bins+1
        xm, ym = [], []
        for k in range(1, bins + 1):
            kk = (b == k)
            if kk.sum() >= min_bin_n:
                xm.append(np.nanmean(x[idx][kk]))
                ym.append(np.nanmean(y[idx][kk]))
        if len(xm) >= 2:
            plt.plot(xm, ym, linewidth=2)

    plt.xlabel("Pseudotime")
    plt.ylabel(score_col)
    plt.title(title if title else f"{score_col} vs pseudotime")
    plt.legend(frameon=False)
    plt.tight_layout()

    if save_prefix is not None:
        png = os.path.join(OUTDIR, f"{save_prefix}.png")
        plt.savefig(png, dpi=300)
        print("saved:", png)

    plt.show()

# 图1：你可以先出两张核心（4→1 和 4_NR→R）
plot_score_vs_pseudotime(ad, "sig_4to1", 'dpt_pseudotime_uniform', title="Signature score: 4→1", save_prefix="Fig1_sig_4to1")
plot_score_vs_pseudotime(ad, "sig_4_NR2R", 'dpt_pseudotime_uniform', title="Signature score: NR→R within cluster 4", save_prefix="Fig1_sig_4_NR2R")

# 如果你也想顺手出 4→2：
plot_score_vs_pseudotime(ad, "sig_4to2", 'dpt_pseudotime_uniform', title="Signature score: 4→2", save_prefix="Fig1_sig_4to2")


In [ ]:
def get_gene_expr(ad, gene, layer="norm_expr_log1p", log1p=True):
    if gene not in ad.var_names:
        raise ValueError(f"{gene} not in ad.var_names")
    X = ad[:, gene].layers[layer] if layer is not None else ad[:, gene].X
    if sparse.issparse(X):
        v = np.asarray(X.todense()).reshape(-1)
    else:
        v = np.asarray(X).reshape(-1)
    if log1p:
        v = np.log1p(v)
    return v

def plot_gene_vs_pseudotime(ad, gene, pseudo_col, group_col="RNR",
                            layer="norm_expr_log1p", log1p=True,
                            bins=60, min_bin_n=25, alpha=0.12,
                            title=None, save_prefix=None):
    x = ad.obs[pseudo_col].values.astype(float)
    y = get_gene_expr(ad, gene, layer=layer, log1p=log1p)
    g = ad.obs[group_col].astype(str).values

    m = np.isfinite(x) & np.isfinite(y) & pd.notna(g)
    x, y, g = x[m], y[m], g[m]

    plt.figure(figsize=(6.2, 4.2))
    for grp in sorted(pd.unique(g)):
        idx = (g == grp)
        plt.scatter(x[idx], y[idx], s=6, alpha=alpha, label=grp)

        edges = np.linspace(np.nanmin(x), np.nanmax(x), bins + 1)
        b = np.digitize(x[idx], edges)
        xm, ym = [], []
        for k in range(1, bins + 1):
            kk = (b == k)
            if kk.sum() >= min_bin_n:
                xm.append(np.nanmean(x[idx][kk]))
                ym.append(np.nanmean(y[idx][kk]))
        if len(xm) >= 2:
            plt.plot(xm, ym, linewidth=2)

    plt.xlabel("Pseudotime")
    ylabel = f"{gene} expression ({'log1p ' if log1p else ''}{layer})"
    plt.ylabel(ylabel)
    plt.title(title if title else f"{gene} vs pseudotime")
    plt.legend(frameon=False)
    plt.tight_layout()

    if save_prefix is not None:
        png = os.path.join(OUTDIR, f"{save_prefix}.png")
        plt.savefig(png, dpi=300)
        print("saved:", png)

    plt.show()

# 你要看的候选：先从最“免疫逻辑强”的开始
genes_to_plot = ["CBLB", "CSK", "ZC3H12A", "RCAN1", "ATM", "G6PD"]

for g in genes_to_plot:
    if g in ad.var_names:
        plot_gene_vs_pseudotime(ad, g, 'dpt_pseudotime_uniform', layer="norm_expr_log1p", log1p=True,
                                title=f"Driver candidate: {g}", save_prefix=f"Fig2_gene_{g}")
    else:
        print("skip (not in var_names):", g)


In [ ]:
def subset_and_plot_cluster(ad, cluster_value, genes):
    ad_sub = ad[ad.obs['leiden_T_0.6'].astype(str) == str(cluster_value)].copy()
    print(f"cluster {cluster_value} n={ad_sub.n_obs}")
    for g in genes:
        if g in ad_sub.var_names:
            plot_gene_vs_pseudotime(ad_sub, g, 'dpt_pseudotime_uniform', layer="norm_expr_log1p", log1p=True,
                                    title=f"{g} in cluster {cluster_value}",
                                    save_prefix=f"Fig2_gene_{g}_cluster{cluster_value}")
        else:
            print("skip:", g)

subset_and_plot_cluster(ad, "4", ["ZC3H12A","CBLB","CSK","RCAN1"])
subset_and_plot_cluster(ad, "1", ["RCAN1","ATM","CBLB","CSK"])


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from scipy import sparse
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

genes_to_test = ["CBLB", "CSK", "ZC3H12A", "RCAN1", "ATM", "G6PD"]

layer_use = "norm_expr_log1p"   # 改成你实际用的
log1p_use = True

pseudo_col = "dpt_pseudotime"
pseudo_col_use = "dpt_pseudotime_uniform"

def get_gene_expr(ad, gene, layer="norm_expr_log1p", log1p=True):
    if gene not in ad.var_names:
        return None
    X = ad[:, gene].layers[layer] if layer is not None else ad[:, gene].X
    if sparse.issparse(X):
        v = np.asarray(X.todense()).reshape(-1)
    else:
        v = np.asarray(X).reshape(-1)
    if log1p:
        v = np.log1p(v)
    return v

def test_mw(expr, grp):
    # Mann–Whitney U: overall NR vs R
    x = expr[grp=="NR"]
    y = expr[grp=="R"]
    if len(x) < 20 or len(y) < 20:
        return np.nan
    _, p = mannwhitneyu(x, y, alternative="two-sided")
    return p

def test_level3_ols(expr, pt, grp):
    # OLS: expr ~ NR + pt (robust SE)
    df = pd.DataFrame({
        "expr": expr,
        "pt": pt,
        "NR": (grp=="NR").astype(int)
    }).replace([np.inf, -np.inf], np.nan).dropna()

    if df.shape[0] < 200:
        return np.nan, np.nan

    X = sm.add_constant(df[["NR", "pt"]])
    m = sm.OLS(df["expr"], X).fit(cov_type="HC3")
    beta = m.params.get("NR", np.nan)
    pval = m.pvalues.get("NR", np.nan)
    return beta, pval

pt = ad.obs[pseudo_col_use].astype(float).values
grp = ad.obs["RNR"].astype(str).values

rows = []
for g in genes_to_test:
    expr = get_gene_expr(ad, g, layer=layer_use, log1p=log1p_use)
    if expr is None:
        print("skip (not found):", g)
        continue

    m = np.isfinite(expr) & np.isfinite(pt) & pd.notna(grp)
    expr2, pt2, grp2 = expr[m], pt[m], grp[m]

    p_mw = test_mw(expr2, grp2)
    beta, p_ols = test_level3_ols(expr2, pt2, grp2)

    rows.append({
        "gene": g,
        "MW_p": p_mw,
        "OLS_beta_NR": beta,
        "OLS_p": p_ols,
        "n_cells": len(expr2),
        "median_NR": np.median(expr2[grp2=="NR"]) if np.any(grp2=="NR") else np.nan,
        "median_R":  np.median(expr2[grp2=="R"])  if np.any(grp2=="R")  else np.nan,
    })

res = pd.DataFrame(rows)

# 多重校正（分别对 MW_p 和 OLS_p 做 FDR）
res["MW_FDR"] = multipletests(res["MW_p"].values, method="fdr_bh")[1]
res["OLS_FDR"] = multipletests(res["OLS_p"].values, method="fdr_bh")[1]

res = res.sort_values("OLS_p", ascending=True)
res_path = os.path.join(out_dir, "level3_NR_vs_R_6genes.csv")
res.to_csv(res_path, index=False)

res, res_path



In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess
import numpy as np

def lowess_line(x, y, frac=0.15, it=0, n_grid=200):
    """
    x,y: 1D arrays
    frac: 平滑程度（越大越平滑，0.1~0.3 常用）
    it: robust iterations（0=普通，1~2=对离群更稳）
    n_grid: 输出网格点数（越大越平滑）
    """
    # 排序 + 去 NaN
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]; y = y[m]
    order = np.argsort(x)
    x = x[order]; y = y[order]

    # LOWESS 返回在原 x 上的拟合值（按 x 排序后）
    fit = lowess(y, x, frac=frac, it=it, return_sorted=True)

    # 可选：插值到均匀网格（线看起来更顺）
    xg = np.linspace(x.min(), x.max(), n_grid)
    yg = np.interp(xg, fit[:,0], fit[:,1])
    return xg, yg

In [ ]:
def plot_gene_vs_pseudotime_with_tests(ad, gene, pseudo_col, outdir,
                                             layer="norm_expr", log1p=True,
                                             group_col="RNR",
                                             alpha=0.10,
                                             lowess_frac=0.18, lowess_it=1):
    expr = get_gene_expr(ad, gene, layer=layer, log1p=log1p)
    if expr is None:
        print("skip:", gene)
        return

    x = ad.obs[pseudo_col].astype(float).values
    g = ad.obs[group_col].astype(str).values

    m = np.isfinite(x) & np.isfinite(expr) & pd.notna(g)
    x, y, gg = x[m], expr[m], g[m]

    # tests
    p_mw = test_mw(y, gg)
    beta, p_ols = test_level3_ols(y, x, gg)

    plt.figure(figsize=(6.2, 4.2))

    for grp_name in ["NR","R"]:
        idx = (gg == grp_name)
        if idx.sum() == 0:
            continue
        plt.scatter(x[idx], y[idx], s=6, alpha=alpha, label=grp_name)

        # LOWESS smooth line
        xg, yg = lowess_line(x[idx], y[idx], frac=lowess_frac, it=lowess_it, n_grid=250)
        plt.plot(xg, yg, linewidth=3)

    # annotate tests
    txt = f"MW p={p_mw:.1e} | OLS beta(NR)={beta:.3f}, p={p_ols:.1e}"
    plt.title(f"Driver candidate: {gene}")
    plt.xlabel("Pseudotime")
    plt.ylabel(f"{gene} expression ({'log1p ' if log1p else ''}{layer})")
    plt.text(0.02, 0.02, txt, transform=plt.gca().transAxes, fontsize=10, va="bottom")

    plt.legend(frameon=False)
    plt.tight_layout()

    png = os.path.join(outdir, f"Fig2_gene_{gene}.png")
    plt.savefig(png, dpi=200)
    plt.show()

    print("saved:", png)

for g in genes_to_test:
    if g in ad.var_names:
        plot_gene_vs_pseudotime_with_tests(
            ad, g, pseudo_col_use, out_dir,
            layer=layer_use, log1p=log1p_use,
            alpha=0.08,
            lowess_frac=0.18,  # 0.12~0.25 可调：越大越平滑
            lowess_it=1        # 1 对离群更稳
        )


In [ ]:
out_dir = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/All_results_0123/perturb_seq"